# Building Agentic RAG Systems

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

> **Validation status.** Preserved outputs are historical evidence from the supplied notebook. Local checks may add their own evidence. Consult the accompanying validation report for the checks run on this revision and the exercises still pending authenticated Colab verification.

## Let's install the dependencies and configure our inference provider

If you haven't installed `smolagents` yet, you can do so by running the following command.

> Colab adaptation: Use Groq instead of the Hugging Face Inference API. All dependencies are installed together to minimize version conflicts.

In [2]:
%pip install -q "smolagents[openai,toolkit]==1.26.0" "requests==2.32.4" ddgs beautifulsoup4 "langchain-text-splitters>=0.3,<2" rank_bm25


Enable `GROQ_API_KEY` in Colab Secrets (key icon), with notebook access switched on. The following cell configures the same Groq model as the previous Code Agents exercise. No Hugging Face token is needed for inference.

Native Groq tool calling is used for retrieval. A successful web answer must quote an actual retrieved URL. A successful BM25 answer must reference a source in the supplied knowledge base.


In [3]:
import os
from time import monotonic, sleep
from uuid import uuid4

from openai import APIConnectionError, APIStatusError, OpenAI, RateLimitError
from smolagents import OpenAIModel, ToolCallingAgent
from smolagents.models import ChatMessageToolCall, ChatMessageToolCallFunction

def read_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None

GROQ_API_KEY = read_secret("GROQ_API_KEY")
GROQ_BASE = "https://api.groq.com/openai/v1"
client = (OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE, timeout=20, max_retries=0)
          if GROQ_API_KEY else None)
MODEL_ID = "openai/gpt-oss-120b"
VISION_MODEL_ID = "qwen/qwen3.8-27b"
TEXT_MODELS = (MODEL_ID, "llama-3.3-70b-versatile", "llama-3.1-8b-instant")
MODEL_REQUEST_LIMIT = 16
model_requests = 0

class GroqAgentModel(OpenAIModel):
    """Use Groq's native tool validation and accept genuine text final answers."""
    def _prepare_completion_kwargs(self, *args, **kwargs):
        policy = getattr(self, "tool_policy", None)
        forced = policy() if policy and kwargs.get("tools_to_call_from") else None
        kwargs["tool_choice"] = (
            {"type": "function", "function": {"name": forced}} if forced else "auto"
        )
        request = super()._prepare_completion_kwargs(*args, **kwargs)
        request.pop("stop", None)
        for message in request["messages"]:
            message.pop("name", None)
        if request.get("tools"):
            request["parallel_tool_calls"] = False
        return request

    def generate(self, messages, *args, tools_to_call_from=None, **kwargs):
        global model_requests
        original_model = self.model_id
        original_options = self.kwargs.copy()
        options = (original_model,) if original_model == VISION_MODEL_ID else TEXT_MODELS
        deadline = monotonic() + 56
        try:
            for attempt in range(2):
                if model_requests >= MODEL_REQUEST_LIMIT or monotonic() >= deadline:
                    raise RuntimeError("Model request budget exhausted.")
                self.model_id = options[min(attempt, len(options) - 1)]
                self.kwargs.pop("reasoning_effort", None)
                if self.model_id.startswith("openai/gpt-oss"):
                    self.kwargs["reasoning_effort"] = "low"
                elif self.model_id == VISION_MODEL_ID:
                    self.kwargs["reasoning_effort"] = "none"
                model_requests += 1
                try:
                    reply = super().generate(
                        messages, *args, tools_to_call_from=tools_to_call_from, **kwargs
                    )
                except (APIStatusError, APIConnectionError) as exc:
                    status = getattr(exc, "status_code", 0) or 0
                    code = getattr(exc, "code", None)
                    recoverable = (status in (404, 429) or status >= 500 or status == 0
                                   or code in ("tool_use_failed", "model_decommissioned"))
                    if attempt or not recoverable:
                        raise RuntimeError(
                            f"Groq request failed for {self.model_id} (HTTP {status or 'connection'}, "
                            f"{code or type(exc).__name__})."
                        ) from None
                    if isinstance(exc, RateLimitError):
                        try:
                            delay = float(exc.response.headers.get("retry-after", "2"))
                        except ValueError:
                            delay = 2
                        if delay > 8 and len(options) == 1:
                            raise RuntimeError(
                                f"Groq vision quota is limited; retry after {delay:g} seconds."
                            ) from None
                        sleep(max(0, min(delay, 8)) if len(options) == 1 else 1)
                    print("Groq request interrupted; making one bounded retry.")
                    continue
                names = {tool.name for tool in tools_to_call_from or ()}
                text = reply.content
                if ("final_answer" in names and not reply.tool_calls
                        and isinstance(text, str) and text.strip()):
                    # smolagents expects a final_answer action. Carry the real text into
                    # that action locally without changing any external tool observations.
                    reply.tool_calls = [ChatMessageToolCall(
                        function=ChatMessageToolCallFunction(
                            name="final_answer", arguments={"answer": text.strip()}),
                        id=f"local_final_{uuid4().hex}", type="function",
                    )]
                if attempt:
                    print("Groq retry completed with", self.model_id)
                return reply
        finally:
            self.model_id = original_model
            self.kwargs = original_options

def make_model(max_tokens=1200, model_id=MODEL_ID):
    if not client:
        raise RuntimeError("Enable notebook access to GROQ_API_KEY in Colab Secrets.")
    return GroqAgentModel(
        model_id=model_id, api_base=GROQ_BASE, api_key=GROQ_API_KEY,
        temperature=0.2, max_completion_tokens=max_tokens, retry=False,
        client_kwargs={"timeout": 20, "max_retries": 0},
    )

def completed_tools(agent):
    return {
        call.name for step in agent.memory.steps
        if getattr(step, "error", None) is None
        for call in (getattr(step, "tool_calls", None) or ())
    }

def run_checked(agent, prompt, required=(), ordered=(), **kwargs):
    def next_tool():
        completed = completed_tools(agent)
        return next((name for name in ordered if name not in completed), None)

    agent.model.tool_policy = next_tool
    try:
        result = agent.run(prompt, return_full_result=True, **kwargs)
    finally:
        agent.model.tool_policy = None
    missing = set(required) - completed_tools(agent)
    if result.state != "success" or missing or not result.output:
        raise RuntimeError(f"Agent exercise incomplete: {result.state}; missing tools: {sorted(missing)}.")
    if result.token_usage:
        print("Agent tokens:", result.token_usage)
    return result.output

# Keep local tool checks available even when Groq credentials are unavailable.
model = make_model() if client else None


## Basic Retrieval with Web Search

Let's build a simple agent that can search the web. This agent will retrieve information and synthesize responses to answer queries. With Agentic RAG, Alfred's agent can:

* Search for latest superhero party trends
* Refine results to include luxury elements
* Synthesize information into a complete plan

Here's how Alfred's agent can achieve this:

> Colab adaptation: The web search tool uses DDGS with automatic backend selection, which was tested successfully in the previous notebook. If a search fails, the tool reports the failure instead of inventing results.

In [4]:
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from urllib.parse import urlparse
from xml.etree import ElementTree

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS

PARTY_SOURCES = (
    "https://www.obrienproductions.com/theme-parties/superhero/",
    "https://www.hipentertainment.com/works/super-heroes",
    "https://www.bizbash.com/event-decor/15-ideas-for-superhero-themed-events",
)
HEADERS = {"User-Agent": "HF-Agents-Course-Colab/1.0 (educational notebook; public sources only)"}

@lru_cache(maxsize=24)
def search_hits(query, fallback_urls=()):
    try:
        hits = DDGS(timeout=8).text(query, max_results=4, backend="auto")
        hits = [hit for hit in hits if urlparse(hit.get("href", "")).scheme == "https"]
        if hits:
            return "DDGS", hits
    except Exception:
        pass

    try:
        response = requests.get(
            "https://www.bing.com/search", params={"q": query, "format": "rss"},
            headers=HEADERS, timeout=(4, 7),
        )
        response.raise_for_status()
        feed = ElementTree.fromstring(response.content)
        hits = [
            {"title": item.findtext("title", ""), "href": item.findtext("link", ""),
             "body": item.findtext("description", "")}
            for item in feed.findall("./channel/item")[:4]
            if item.findtext("link", "").startswith("https://")
        ]
        if hits:
            return "Bing RSS", hits
    except (requests.RequestException, ElementTree.ParseError):
        pass

    def read_source(url):
        try:
            response = requests.get(url, headers=HEADERS, timeout=(4, 7))
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")
            title = soup.title.get_text(" ", strip=True) if soup.title else ""
            if not title:
                return None
            meta = soup.find("meta", attrs={"name": "description"})
            summary = meta.get("content", "") if meta else ""
            if not summary:
                summary = " ".join(p.get_text(" ", strip=True) for p in soup.select("main p, article p")[:5])
            return {"title": title, "href": response.url, "body": summary[:400]}
        except requests.RequestException:
            return None

    if fallback_urls:
        with ThreadPoolExecutor(max_workers=3) as workers:
            hits = [hit for hit in workers.map(read_source, fallback_urls) if hit]
        if hits:
            return "live reads of curated public sources", hits
    return "unavailable", []

def format_hits(backend, hits):
    entries = [
        f"{hit['title']} | {hit['href']} | {hit['body'][:350]}"
        for hit in hits
    ]
    if not entries:
        return "No sources retrieved. Report this limitation."
    return f"Retrieval: {backend}\n" + "\n".join(entries)

from ddgs import DDGS
from smolagents import ToolCallingAgent, Tool


class WebSearchTool(Tool):
    name = "web_search"
    description = (
        "Searches the web for up-to-date information and returns titles, URLs, "
        "and snippets. If no sources are available, reports the search failure."
    )
    inputs = {
        "query": {"type": "string", "description": "The search query."}
    }
    output_type = "string"


    def forward(self, query: str) -> str:
        backend, hits = search_hits(query, PARTY_SOURCES)
        retrieved_urls.update(hit["href"] for hit in hits)
        return format_hits(backend, hits)

retrieved_urls = set()

# Initialize the search tool
search_tool = WebSearchTool()

# Initialize the agent with the Groq model from the setup cell
agent = ToolCallingAgent(model=model, tools=[search_tool], max_steps=4)

# Example usage
response = run_checked(
    agent,
    "Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering. Call web_search and cite only retrieved URLs.",
    required=("web_search",), ordered=("web_search",),
)
if not any(url in str(response) for url in retrieved_urls):
    raise RuntimeError("No retrieved URL was cited in the web retrieval answer.")
print(response)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering. Call        │
│ web_search and cite only retrieved URLs.                                                                        │
│                                                                                                                 │
╰─ GroqAgentModel - openai/gpt-oss-120b ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'luxury superhero themed party ideas decorations           │
│ entertainment catering'}                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Retrieval: Bing RSS
LUXURY Definition & Meaning - Merriam-Webster | https://www.merriam-webster.com/dictionary/luxury | The meaning of 
LUXURY is a condition of abundance or great ease and comfort : sumptuous environment. How to use luxury in a 
sentence.
Luxury - Wikipedia | https://en.wikipedia.org/wiki/Luxury | This disambiguation page lists articles associated with
the title Luxury. If an internal link incorrectly led you here, you may wish to change the link to point directly 
to the intended article.
LUXURY | English meaning - Cambridge Dictionary | https://dictionary.cambridge.org/dictionary/english/luxury | 
LUXURY definition: 1. great comfort, especially as provided by expensive and beautiful things: 2. something 
expensive…. Learn more.
Luxury Escapes - Handpicked Escapes at the Best Prices on Earth | https://luxuryescapes.com/us | Find the latest 
luxury travel ideas & exclusive special offers at insider prices. Save on vacation travel & accommodations packages
with Luxury Escapes.

[Step 1: Duration 3.80 seconds| Input tokens: 1,051 | Output tokens: 46]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (5 outputs left out)

Observations: **Luxury Superhero‑Themed Party Ideas**

| Category | Luxury‑Level Ideas | Example Sources |
|----------|-------------------|-----------------|
| **Decorations** | • Black‑tie entrance with a red‑carpet “Hall of Heroes” and illuminated superhero logos on the 
floor.<br>• Custom‑printed silk tablecloths in deep navy with gold‑foil emblems of iconic symbols (e.g., “S”, “A”, 
“⚡”).<br>• Premium balloon arches in metallic gold and royal blue, topped with LED‑lit superhero silhouettes.<br>•
Crystal chandeliers draped with subtle cape‑style fabric panels and subtle glitter‑dust effects. | 
https://nationaleventpros.com/creative-event-themes-superhero-party/ |
| **Entertainment** | • Professional stunt performers dressed as Marvel/D.C. heroes for a live “heroic showcase” 
(martial‑arts, acrobatics).<br>• Interactive VR stations where guests can experience a superhero mission in a 
high‑end headset.<br>• A live orchestra playing heroic movie scores, with a DJ mixing in modern beats for a “heroic
dance floor.”<br>• Luxury photo‑booth with green‑screen backdrops of iconic city skylines, printed on glossy metal 
frames. | https://hostessmagazine.com/superhero-decorations/ |
| **Catering** | • Gourmet “hero” canapés: truffle‑infused mini‑sliders (Captain America), smoked salmon 
“Kryptonite” blinis (Superman), gold‑leaf chocolate mousse “Infinity Stone” bites.<br>• Signature cocktail menu 
with names like “The Dark Knight” (bourbon, blackberries, smoked ice) and “Wonder Woman’s Lasso” (sparkling rosé, 
edible glitter).<br>• Premium dessert bar featuring a multi‑layer “Justice League” cake with mirrored fondant and 
edible metallic accents.<br>• Champagne tower styled as a superhero emblem, served by professionally trained staff.
| https://nationaleventpros.com/creative-event-themes-superhero-party/ |
| **Additional Luxury Touches** | • Custom‑designed invitation scrolls with embossed superhero insignia and QR 
codes linking to a private event app.<br>• valet parking with attendants in sleek superhero‑themed uniforms.<br>• 
Gift bags containing designer merchandise (e.g., silk capes, monogrammed utility belts, limited‑edition collectible
figurines). | https://hostessmagazine.com/superhero-decorations/ |

**Summary**

To create a truly luxurious superhero‑themed party, combine high‑end décor (metallic fabrics, crystal lighting, 
custom balloon arches), premium entertainment (stunt shows, VR experiences, live orchestral music), and upscale 
catering (gourmet hero‑inspired bites, signature cocktails, champagne tower). Use the cited sources for detailed 
inspiration and visual examples.

Final answer: **Luxury Superhero‑Themed Party Ideas**

| Category | Luxury‑Level Ideas | Example Sources |
|----------|-------------------|-----------------|
| **Decorations** | • Black‑tie entrance with a red‑carpet “Hall of Heroes” and illuminated superhero logos on the 
floor.<br>• Custom‑printed silk tablecloths in deep navy with gold‑foil emblems of iconic symbols (e.g., “S”, “A”, 
“⚡”).<br>• Premium balloon arches in metallic gold and royal blue, topped with LED‑lit superhero silhouettes.<br>•
Crystal chandeliers draped with subtle cape‑style fabric panels and subtle glitter‑dust effects. | 
https://nationaleventpros.com/creative-event-themes-superhero-party/ |
| **Entertainment** | • Professional stunt performers dressed as Marvel/D.C. heroes for a live “heroic showcase” 
(martial‑arts, acrobatics).<br>• Interactive VR stations where guests can experience a superhero mission in a 
high‑end headset.<br>• A live orchestra playing heroic movie scores, with a DJ mixing in modern beats for a “heroic
dance floor.”<br>• Luxury photo‑booth with green‑screen backdrops of iconic city skylines, printed on glossy metal 
frames. | https://hostessmagazine.com/superhero-decorations/ |
| **Catering** | • Gourmet “hero” canapés: truffle‑infused mini‑sliders (Captain America), smoked salmon 
“Kryptonite” blinis (Superman), gold‑leaf chocolate mousse “Infinity Stone” bites.<br>• Signature cocktail menu 
with names like “The Dark Knight” (bourbon, blackberries, smoked ice) and “Wonder Woman’s Lasso” (sparkling rosé, 
edible glitter).<br>• Premium dessert bar featuring a multi‑layer “Justice League” cake with mirrored fondant and 
edible metallic accents.<br>• Champagne tower styled as a superhero emblem, served by professionally trained staff.
| https://nationaleventpros.com/creative-event-themes-superhero-party/ |
| **Additional Luxury Touches** | • Custom‑designed invitation scrolls with embossed superhero insignia and QR 
codes linking to a private event app.<br>• valet parking with attendants in sleek superhero‑themed uniforms.<br>• 
Gift bags containing designer merchandise (e.g., silk capes, monogrammed utility belts, limited‑edition collectible
figurines). | https://hostessmagazine.com/superhero-decorations/ |

**Summary**

To create a truly luxurious superhero‑themed party, combine high‑end décor (metallic fabrics, crystal lighting, 
custom balloon arches), premium entertainment (stunt shows, VR experiences, live orchestral music), and upscale 
catering (gourmet hero‑inspired bites, signature cocktails, champagne tower). Use the cited sources for detailed 
inspiration and visual examples.

[Step 3: Duration 1.70 seconds| Input tokens: 4,211 | Output tokens: 726]

Agent tokens: TokenUsage(input_tokens=4211, output_tokens=726, total_tokens=4937)
**Luxury Superhero‑Themed Party Ideas**

| Category | Luxury‑Level Ideas | Example Sources |
|----------|-------------------|-----------------|
| **Decorations** | • Black‑tie entrance with a red‑carpet “Hall of Heroes” and illuminated superhero logos on the floor.<br>• Custom‑printed silk tablecloths in deep navy with gold‑foil emblems of iconic symbols (e.g., “S”, “A”, “⚡”).<br>• Premium balloon arches in metallic gold and royal blue, topped with LED‑lit superhero silhouettes.<br>• Crystal chandeliers draped with subtle cape‑style fabric panels and subtle glitter‑dust effects. | https://nationaleventpros.com/creative-event-themes-superhero-party/ |
| **Entertainment** | • Professional stunt performers dressed as Marvel/D.C. heroes for a live “heroic showcase” (martial‑arts, acrobatics).<br>• Interactive VR stations where guests can experience a superhero mission in a high‑end headset.<br>• A live orches

The agent follows this process:

1. **Analyzes the Request:** Alfred’s agent identifies the key elements of the query—luxury superhero-themed party planning, with focus on decor, entertainment, and catering.
2. **Performs Retrieval:**  The agent leverages WebSearchTool to search for the most relevant and up-to-date information, ensuring it aligns with Alfred’s refined preferences for a luxurious event.
3. **Synthesizes Information:** After gathering the results, the agent processes them into a cohesive, actionable plan for Alfred, covering all aspects of the party.
4. **Stores for Future Reference:** The agent stores the retrieved information for easy access when planning future events, optimizing efficiency in subsequent tasks.

> Note: The agent's run history retains retrieved information during execution. This example does not configure a persistent knowledge store across separate runs.

## Custom Knowledge Base Tool

For specialized tasks, a custom knowledge base can be invaluable. Let's create a tool that queries a vector database of technical documentation or specialized knowledge. Using semantic search, the agent can find the most relevant information for Alfred's needs.

This approach combines predefined knowledge with semantic search to provide context-aware solutions for event planning. With specialized knowledge access, Alfred can perfect every detail of the party.

Install the dependencies first and run!

> Implementation note: This exercise uses BM25, a lexical ranking algorithm, rather than an embedding-based vector database. The original course's example and knowledge base are retained.

In [5]:
# The dependencies were installed together in the first cell.
from importlib.metadata import version

for package in ("langchain-text-splitters", "rank-bm25"):
    print(f"{package}: {version(package)}")


langchain-text-splitters: 1.1.3
rank-bm25: 0.2.2


In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from smolagents import Tool
import re
from rank_bm25 import BM25Okapi
from smolagents import ToolCallingAgent

class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = "Uses BM25 lexical search to retrieve relevant party planning ideas for Alfred’s superhero-themed party at Wayne Manor."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be a query related to party planning or superhero themes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)

        self.docs = list(docs)
        self.retriever = BM25Okapi([
            self.tokenize(f"{doc.metadata['source']} {doc.page_content}") for doc in self.docs
        ])

    @staticmethod
    def tokenize(text):
        return re.findall(r"[a-z0-9]+", text.casefold())

    def forward(self, query: str) -> str:
        hits = self.retriever.get_top_n(self.tokenize(query), self.docs, n=min(4, len(self.docs)))
        return "\n\n".join(
            f"[{doc.metadata['source']}] {doc.page_content}" for doc in hits
        )

# Simulate a knowledge base about party planning
party_ideas = [
    {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
    {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
    {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
    {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
    {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"}
]

source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

# Split the documents into smaller chunks for more efficient search
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
docs_processed = text_splitter.split_documents(source_docs)

# Create the retriever tool
party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

print(party_planning_retriever("superhero entertainment catering decoration"))

# Initialize the agent
agent = ToolCallingAgent(tools=[party_planning_retriever], model=model, max_steps=4)

# Example usage
response = run_checked(
    agent,
    "Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration. Call party_planning_retriever with a query that covers these categories, and cite the knowledge base source labels.",
    required=("party_planning_retriever",), ordered=("party_planning_retriever",),
)

if not any(doc["source"] in str(response) for doc in party_ideas):
    raise RuntimeError("The BM25 answer did not cite a knowledge-base source label.")
print(response)


[Catering Ideas] For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'

[Decoration Ideas] Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.

[Entertainment Ideas] Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.

[Entertainment Ideas] Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration. Call         │
│ party_planning_retriever with a query that covers these categories, and cite the knowledge base source labels.  │
│                                                                                                                 │
╰─ GroqAgentModel - openai/gpt-oss-120b ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'party_planning_retriever' with arguments: {'query': 'luxury superhero-themed party entertainment │
│ catering decoration ideas'}                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |Party Ideas 1] A superhero-themed masquerade ball with luxury decor, including gold accents and 
velvet curtains.

|Catering Ideas] For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron 
Man's Power Steak.'

|Decoration Ideas] Decorate with iconic superhero logos and projections of Gotham and other superhero cities around
the venue.

|Entertainment Ideas] Interactive experiences with VR where guests can engage in superhero simulations or compete 
in themed games.

[Step 1: Duration 0.42 seconds| Input tokens: 1,088 | Output tokens: 53]

... (2 outputs left out)

Observations: {
  "name": "final_answer",
  "arguments": {
    "answer": "Based on the retrieved ideas, a luxury superhero‑themed party can include:\n\n**Entertainment** – 
Interactive VR superhero simulations and themed games where guests can compete as their favorite 
heroes.\n\n**Catering** – Gourmet dishes named after superheroes, such as “The Hulk’s Green Smoothie” and “Iron 
Man’s Power Steak,” offering a high‑end twist on character‑themed food.\n\n**Decoration** – A masquerade‑ball 
atmosphere with luxury décor: gold accents, velvet curtains, iconic superhero logos, and projections of Gotham and 
other superhero cities around the venue.\n\nThese suggestions come from the party planning knowledge base (source 
labels: |Party Ideas 1], |Catering Ideas], |Decoration Ideas], |Entertainment Ideas])."
  }
}

Final answer: {
  "name": "final_answer",
  "arguments": {
    "answer": "Based on the retrieved ideas, a luxury superhero‑themed party can include:\n\n**Entertainment** – 
Interactive VR superhero simulations and themed games where guests can compete as their favorite 
heroes.\n\n**Catering** – Gourmet dishes named after superheroes, such as “The Hulk’s Green Smoothie” and “Iron 
Man’s Power Steak,” offering a high‑end twist on character‑themed food.\n\n**Decoration** – A masquerade‑ball 
atmosphere with luxury décor: gold accents, velvet curtains, iconic superhero logos, and projections of Gotham and 
other superhero cities around the venue.\n\nThese suggestions come from the party planning knowledge base (source 
labels: [Party Ideas 1], [Catering Ideas], [Decoration Ideas], [Entertainment Ideas])."
  }
}

[Step 2: Duration 0.83 seconds| Input tokens: 2,354 | Output tokens: 254]

Agent tokens: TokenUsage(input_tokens=2354, output_tokens=254, total_tokens=2608)
{
  "name": "final_answer",
  "arguments": {
    "answer": "Based on the retrieved ideas, a luxury superhero‑themed party can include:\n\n**Entertainment** – Interactive VR superhero simulations and themed games where guests can compete as their favorite heroes.\n\n**Catering** – Gourmet dishes named after superheroes, such as “The Hulk’s Green Smoothie” and “Iron Man’s Power Steak,” offering a high‑end twist on character‑themed food.\n\n**Decoration** – A masquerade‑ball atmosphere with luxury décor: gold accents, velvet curtains, iconic superhero logos, and projections of Gotham and other superhero cities around the venue.\n\nThese suggestions come from the party planning knowledge base (source labels: [Party Ideas 1], [Catering Ideas], [Decoration Ideas], [Entertainment Ideas])."
  }
}


This enhanced agent can:
1. First check the documentation for relevant information
2. Combine insights from the knowledge base
3. Maintain conversation context through memory

> Save this notebook with your own execution outputs. The sample knowledge base is supplied by the course; live web results are retrieved at runtime.

## Local component verification

The following checks ran without model inference. They record actual tool and data outputs. Run the exercises above in authenticated Colab to obtain new agent results.

In [7]:
search_tool = WebSearchTool()
query = "superhero theme event catering decor entertainment"
evidence = search_tool(query)
if not retrieved_urls:
    raise RuntimeError("Public search returned no usable URLs.")
print(evidence)
print("\nActual local BM25 matches:")
print(party_planning_retriever("superhero entertainment catering decoration"))

Retrieval: DDGS
15 Ideas for Superhero-Themed Events - BizBash | https://www.bizbash.com/event-decor/15-ideas-for-superhero-themed-events | Jul 19, 2012 · Given the popularity of other films and TV shows about comic book characters and people with superhuman powers, it's no surprise hosts have been incorporating superhero themes into their events. Here's a look at different ideas for adding cartoon-like visuals and references to caped crusaders at your next affair. Exterior Illustratio
Superhero Event Ideas - Pinterest | https://www.pinterest.com/ideas/superhero-event-ideas/901981400666/ | Design Superhero Themed Party. Marvel Fan Event Inspiration. Superhero Themed Event.
Creative Event Themes: Superhero Party | https://nationaleventpros.com/creative-event-themes-superhero-party/ | Invite your guests to dress up as their favorite superhero, and give out prizes for the best costumes. Then, ask guests to come up with superpowers for each other, recognizing the contributions of their tea